---
title: "DRG Cleaning v2"

author: "Carlos Resurreccion"

date: "2024-10-21"

---

Libraries

In [1]:
# Update submodules like the grouper
system("git submodule update --init --recursive")

# Delete all R objects and run garbage collection so we start with a clean slate
rm(list = ls())
gc()

# List, install (if applicable), and load packages
## Required packages
required_packages <- c(
  "data.table", "here", "tictoc", "stringr", "stringi", "lubridate",
  "profvis", "hash", "future", "future.apply", "knitr", "htmlwidgets",
  "parallelly", "stringdist", "parallel", "reticulate", "bigrquery",
  "jsonlite", "googleCloudStorageR", "haven"
  # , "docstring", "progress" # Comma is here so if I uncomment this line it
  # automatically works without having to type or delete a comma after haven
)
## Install and load required packages
lapply(required_packages, function(package) {
  if (!require(package, character.only = TRUE)) {
    install.packages(package, dependencies = TRUE)
  }
  library(package, character.only = TRUE)
})


R Scripts

In [2]:
scripts <- list( # List of scripts to source
  lib_params = "00_v2_params-fpaths.R",
  cleaning = "01_v2_cleaning-functions.R",
  clinical = "02_v2_clinical-functions.R",
  timing_debug = "03_v2_timing-debug-functions.R",
  summary = "04_v2_summary-functions.R",
  io = "05_v2_io-functions.R"
)

# Loop to source above scripts
for (script in scripts) source(here("data-cleaning/r_scripts_v2", script))


Parameters

Change which year to process in 
`~/drg-pipeline/data-cleaning/cache/year_to_load.txt`

Change other rarely touched parameters in 
`~/drg-pipeline/data-cleaning/r_scripts/00_v2_params-fpaths.R`

In [ ]:
# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample <- TRUE
# TODO: Add description here
to_write <- TRUE
# TODO: Add description here
to_flush <- FALSE
# TODO: Add description here
to_parallel <- TRUE


Load Full Claims from GCS

In [ ]:
# Load raw claims from GCS only if they don't exist on the VM yet
for (year in 2018:2023) {
  # Assign the correct file extension based on the year
  file_type <- if (year %in% c(2022:2023)) ".tsv" else ".csv"
  file_name <- paste0(full_claims_prefix, year, file_type)
  bq_name <- paste0(full_claims_bq_prefix, year, file_type)

  # Check if the file exists in the target directory
  file_path <- here(raw_claims_path, file_name)
  exists <- file.exists(file_path)

  # If the file does not exist, run the gsutil cp command
  if (!exists) {
    if (!is.null(gcp_proj) && gcp_proj == "drg-pipeline") {
      system(paste0("cd .. && gsutil cp gs://phic-claims-raw/", bq_name, " ", raw_claims_path),
        intern = FALSE, ignore.stderr = FALSE
      )
    } else {
      stop("Error: GCP Project is not null and is not drg-pipeline")
    }
  } else {
    message(paste("File", file_name, "already exists in the target directory. Skipping download.\n"))
  }
}


Load Mapping Data

In [ ]:
query_bq_to_dt <- function(query, max_bq_rows = Inf) {
  tryCatch(
    dt <- as.data.table(
      bq_table_download(bq_project_query(gcp_proj, query), n_max = max_bq_rows)
    ),
    error = function(e) {
      stop(paste("Error querying BigQuery:", e$message))
    }
  )
  return(dt)
}

# 1. Query and load `grouper_v5.proc`
proc_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.proc`")
proc <- query_bq_to_dt(proc_query)
proc[, CODE := as.character(CODE)]

# 2. Query and load `phic.acr_rvs_map`
rvs_icd9_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_rvs_map`")
rvs_icd9 <- query_bq_to_dt(rvs_icd9_query)

# Convert rvs to character and handle icd9cm conversion
rvs_icd9 <- rvs_icd9[, .(
  rvs = as.character(rvs),
  icd9cm = as.character(as.numeric(icd9cm) * 100)
)]

# Merge with proc to classify by DRGUSE
rvs_icd9 <- merge(
  rvs_icd9,
  proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)

# Remove DRGUSE and filter out NAs
rvs_icd9 <- rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE][
  !is.na(rvs) & !is.na(icd9cm), -"DRGUSE"
]

# 3. Query and load `phic.acr_procedure`
acr_rvs_query <- paste0("SELECT * FROM `", gcp_proj, ".phic.acr_procedure`")
acr_rvs <- query_bq_to_dt(acr_rvs_query)

# 4. Query and load `grouper_v5.i10`
i10_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10`")
tdrg_icd10 <- query_bq_to_dt(i10_query)
setkey(tdrg_icd10, "CODE")

# Subset and assign to acc_pdx
acc_pdx <- unique(tdrg_icd10[ACCPDX == "Y", CODE])

# 5. Query and load `icd.phl_icd10`
phl_icd10_query <- paste0("SELECT * FROM `", gcp_proj, ".icd.phl_icd10`")
phl_icd10 <- query_bq_to_dt(phl_icd10_query)

# Filter and process neoplasms
neoplasms_dt_actual <- as.data.table(phl_icd10[
  grepl("/", icd10), .(icd10)
][, icd10 := sapply(strsplit(icd10, ","), function(x) trimws(x[2]))])

# 6. Query and load `drg-pipeline.grouper_v5.i10vx`
i10vx_query <- paste0("SELECT * FROM `", gcp_proj, ".grouper_v5.i10vx`")
i10vx <- query_bq_to_dt(i10vx_query)
setkey(i10vx, "code")
acc_icd <- unique(i10vx[, code])

# 7. Query and load `drg-pipeline.hci.temp_hci`
hci_query <- paste0("SELECT * FROM `", gcp_proj, ".hci.temp_hci`")
hci <- query_bq_to_dt(hci_query)


In [ ]:
# Call the main function with or without profvis
# Start main execution logic
# Step 1: Read the header of the full claims file
full_header <<- fread(
  file = full_claims_file,
  nrows = 1, colClasses = "character",
  header = TRUE, encoding = encode # , sep = separator
)

partial_file <- here(raw_claims_parts_path, paste0(
  full_claims_prefix, year_to_load,
  "_part_", sprintf("%02d", split_parts),
  "_of_", split_parts, ".rds"
))

# Step 2: Check if the split part file already exists. If not, read the full claims file.
if (!file.exists(partial_file)) {
  # Read the full file into memory
  full_file <<- fread(
    file = full_claims_file, colClasses = "character",
    header = TRUE, encoding = encode # , sep = separator
  )
}

# Step 3: Split the file into parts and save them
start_time <<- Sys.time() # Record start time
for (split_loop_part in 1:split_parts) {
  function(split_loop_part) {
    # Calculate how many rows per part
    rows_per_part <- ceiling(total_rows / split_parts)

    chunk_file <- here(raw_claims_parts_path, paste0(
      full_claims_prefix, year_to_load,
      "_part_", sprintf("%02d", split_loop_part),
      "_of_", split_parts, ".rds"
    ))

    # Only process if the part does not already exist
    if (!file.exists(chunk_file)) {
      # Determine start and end rows for this chunk
      start_row <- (split_loop_part - 1) * rows_per_part + 1
      end_row <- min(split_loop_part * rows_per_part, total_rows)

      # Extract chunk of data for processing
      chunk_dt <- full_file[start_row:end_row]

      # Save the chunk as an RDS file
      saveRDS(chunk_dt, chunk_file, compress = TRUE)
      rm(chunk_dt)
      gc()

      # Save processing time for this part
      split_processing_times[[split_loop_part]] <- as.numeric(
        difftime(Sys.time(), start_time, units = "secs")
      )

      # Function to print status updates using lubridate
      print_status_update <- function(status_part, split_parts, processing_times, phase) {
        #' @title Print Status Update
        #' @description Print the status update and estimated time remaining.
        #' @param status_part integer. The current status_part number.
        #' @param split_parts integer. Total number of parts.
        #' @param processing_times numeric. Array of processing times for each
        #' status_part.

        # Calculate elapsed time and averages
        elapsed_time <- sum(processing_times[1:status_part])
        avg_time_per_part <- elapsed_time / status_part
        estimated_total_time <- avg_time_per_part * split_parts
        estimated_remaining_time <- estimated_total_time - elapsed_time

        # Convert time to period (using lubridate)
        convert_to_hr_min_sec <- function(seconds) {
          # Round seconds to the nearest whole number
          period <- seconds_to_period(round(seconds))
          return(period)
        }

        # Calculate elapsed and remaining time
        elapsed <- convert_to_hr_min_sec(elapsed_time)
        remaining <- convert_to_hr_min_sec(estimated_remaining_time)

        # Format period to string
        format_time <- function(period) {
          # Extract components
          h <- hour(period)
          m <- minute(period)
          s <- second(period)

          # Construct time string with labels
          time_components <- c()
          if (h > 0) time_components <- c(time_components, paste0(h, "h"))
          if (m > 0 || h > 0) time_components <- c(time_components, paste0(m, "m"))
          time_components <- c(time_components, paste0(s, "s"))

          # Join components and return
          time_str <- paste(time_components, collapse = " ")
          return(trimws(time_str))
        }

        elapsed_str <- format_time(elapsed)
        remaining_str <- format_time(remaining)

        # Determine when to print the status update
        if (phase == "split") {
          if (avg_time_per_part >= 4) {
            # Print status updates for every status_part
            cat(sprintf(
              "\rFinished splitting %d of %d parts in %s (ETA %s)       ",
              status_part, split_parts, elapsed_str, remaining_str
            ))
            flush.console()
          } else if (avg_time_per_part < 4 && status_part %% 5 == 0) {
            # Print status updates for every 5th, 10th, 15th status_part
            cat(sprintf(
              "\rFinished splitting %d of %d parts in %s (ETA %s)       ",
              status_part, split_parts, elapsed_str, remaining_str
            ))
            flush.console()
          }
        } else if (phase == "clean") {
          if (avg_time_per_part >= 4) {
            # Print status updates for every status_part
            cat(sprintf(
              "\rFinished cleaning %d of %d parts in %s (ETA %s)       ",
              status_part, split_parts, elapsed_str, remaining_str
            ))
            flush.console()
          } else if (avg_time_per_part < 4 && status_part %% 5 == 0) {
            # Print status updates for every 5th, 10th, 15th status_part
            cat(sprintf(
              "\rFinished cleaning %d of %d parts in %s (ETA %s)       ",
              status_part, split_parts, elapsed_str, remaining_str
            ))
            flush.console()
          }
        }
      }

      # Print status update and estimate remaining time
      print_status_update(split_loop_part, split_parts, split_processing_times, "split")
    }
  }
}

# Step 4: DEPRECATED


In [ ]:
# Step 5: Loop through each part and process the partial files
for (loop_part in 1:split_parts) {
  start_time <- Sys.time() # Record start time for processing
  partial_claims_file <<- here(raw_claims_parts_path, paste0(
    full_claims_prefix, year_to_load,
    "_part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
  ))

  # Step 6: Handle sampling logic if applicable
  if (to_sample) {
    sampled_claims_file <<- here(raw_claims_samples_path, paste0(
      "sampled_claims_", year_to_load, "_", sample_size,
      "_part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
    ))

    ensure_sample_files_exist <- function(sample_part) {
      if (!file.exists(sampled_claims_file)) {
        partial_file_for_sampling <- here(raw_claims_parts_path, paste0(
          full_claims_prefix, year_to_load,
          "_part_", sprintf("%02d", sample_part), "_of_", split_parts, ".rds"
        ))
        dt <- readRDS(partial_file_for_sampling)
        dt <- dt[sample(.N, min(sample_size, .N))]
        # setnames(dt, colnames(full_header))
        saveRDS(dt, sampled_claims_file, compress = FALSE)
      }
    }

    # See above
    ensure_sample_files_exist(loop_part) # Ensure sample files exist
  }

  read_appropriate_file <- function(read_part, to_sample = to_sample) {
    chunk_file <- if (to_sample) {
      sampled_claims_file
    } else {
      here(raw_claims_parts_path, paste0(
        full_claims_prefix, year_to_load,
        "_part_", sprintf("%02d", read_part), "_of_", split_parts, ".rds"
      ))
    }

    dt <- readRDS(chunk_file)

    available_columns <- colnames(dt)

    # Drop columns
    if (any(drop_cols %in% available_columns)) {
      dt <- dt[, (drop_cols) := NULL]
    }

    if (any(drop_cols_manual %in% available_columns)) {
      dt <- dt[, (drop_cols_manual) := NULL]
    }

    replace_result <- replace_empty_with_na(dt = dt, to_view_checks)
    dt <- replace_result$return_data
    replacement_summary <- replace_result$return_replacement_summary

    if (to_debug) print(head(dt), 2) # debug

    ## Apply column classes only to the columns that exist in the data
    col_classes <- sapply(available_columns, function(col) {
      if (col %in% unlist(expected_types["character"])) {
        return("character")
      }
      if (col %in% unlist(expected_types["integer"])) {
        return("integer")
      }
      if (col %in% unlist(expected_types["factor"])) {
        return("factor")
      }
      if (col %in% unlist(expected_types["numeric"])) {
        return("numeric")
      }
    })

    # Cast column types with checks
    for (col in names(col_classes)) {
      original_values <- dt[[col]]

      dt[[col]] <- switch(col_classes[[col]],
        "character" = as.character(dt[[col]]),
        "factor" = {
          levels <- unique(dt[[col]])
          as.factor(dt[[col]])
        },
        "integer" = {
          suppressWarnings(as.integer(dt[[col]]))
        },
        "numeric" = {
          suppressWarnings(as.numeric(dt[[col]]))
        },
        dt[[col]] # Default case: no conversion if unrecognized type
      )

      # Check for NA coercion
      coerced_to_na <- which(is.na(dt[[col]]) & !is.na(original_values))
      if (length(coerced_to_na) > 0) {
        cat(sprintf(
          "Column '%s' coerced %d values to NA. First few original values: %s\n",
          col, length(coerced_to_na), paste(original_values[coerced_to_na][1:5],
            collapse = ", "
          )
        ))
      }
    }

    nrow_start[[read_part]] <<- nrow(dt)

    return(
      list(
        read_result_dt = dt,
        read_result_replacement_summary = replacement_summary
      )
    )
  }

  # Step 7: Read the appropriate file (sample or full)
  # See above
  read_result <- read_appropriate_file(loop_part)
  read_in_dt <- read_result$read_result_dt # The data to process
  read_in_replacement_summary <- read_result$read_result_replacement_summary # Any replacements summary

  # Step 8: Split the data into chunks for parallel processing
  chunk_size <- ceiling(nrow(read_in_dt) / nthreads)
  chunks <- split(read_in_dt, rep(1:nthreads, each = chunk_size, length.out = nrow(read_in_dt)))

  # Define process_chunk (not to be confused with process_part) that processes each part in nthreads chunks
  process_chunk <- function(chunk,
                            to_view_checks = to_view_checks,
                            rvs_icd9 = rvs_icd9,
                            tdrg_icd10 = tdrg_icd10,
                            acc_pdx = acc_pdx) {
    # Step 1: DEPRECATED

    # Define main clean data function, which does majority of the data cleaning on the claims file
    clean_data <- function(dt) {
      # Rename columns based on column_mappings
      available_columns <- colnames(dt)

      # Convert source year to integer
      if ("ADMISSION_YEAR" %in% available_columns) {
        # do nothing
      } else {
        dt[, SRC_YR := as.integer(year_to_load)]
      }

      dt[, is_covid := FALSE]

      # Identify the columns to rename based on the mapping
      old_names <- available_columns[available_columns %in% names(column_mappings)]
      new_names <- sapply(old_names, function(col) column_mappings[[col]])

      # Rename the columns in the data.table
      setnames(dt, old = old_names, new = new_names)

      # Check if renaming was successful
      rename_success <- all(new_names %in% available_columns)

      dt[, id_series := trimws(id_series)]

      dt[, c1 := clin_c1]
      dt[, c2 := clin_c2]

      # Convert time_adm and time_dis for 2022-2023 format
      # Strip fractional seconds and handle AM/PM conversion properly using as.POSIXct()
      dt[, time_adm := ifelse(grepl("AM|PM", time_adm),
        format(as.POSIXct(sub("\\.\\d+ ", " ", time_adm), format = "%m/%d/%Y %I:%M:%S %p"), "%H:%M"),
        time_adm
      )]

      dt[, time_dis := ifelse(grepl("AM|PM", time_dis),
        format(as.POSIXct(sub("\\.\\d+ ", " ", time_dis), format = "%m/%d/%Y %I:%M:%S %p"), "%H:%M"),
        time_dis
      )]

      collapse_and_clean_icd_rvs <- function(dt) {
        ## Collapses and cleans ICD and RVS columns in a data.table
        # dt: input data.table containing ICD and RVS columns
        available_columns <- colnames(dt)

        # Dynamically detect which clin_icd columns exist
        icd_cols <- grep("^clin_icd\\d+$", available_columns, value = TRUE)
        if (length(icd_cols) > 0) {
          # Collapse the ICD codes, whether from multiple columns or a single column
          dt[, clin_icd := collapse_columns(mget(icd_cols), na_like_strings)]
          dt[, (icd_cols) := NULL] # Remove the individual columns after collapsing
        }

        # Dynamically detect which clin_rvs columns exist
        rvs_cols <- grep("^clin_rvs\\d+$", available_columns, value = TRUE)
        if (length(rvs_cols) > 0) {
          # Collapse the RVS codes, whether from multiple columns or a single column
          dt[, clin_rvs := collapse_columns(mget(rvs_cols), na_like_strings)]
          dt[, (rvs_cols) := NULL] # Remove the individual columns after collapsing
        }

        # Handle any lumped ICD codes by splitting them if clin_icd exists
        if ("clin_icd" %in% available_columns) {
          dt[, clin_icd := remove_lumped_icd_codes(clin_icd)] # Apply cleaning for lumped codes
          dt[, clin_icd := split_to_vector(clin_icd)] # Convert cleaned string to vector
        }

        # Handle any lumped RVS codes by splitting them if clin_rvs exists
        if ("clin_rvs" %in% available_columns) {
          # Assuming there is a `remove_lumped_rvs_codes` function, apply it here.
          # dt[, clin_rvs := remove_lumped_rvs_codes(clin_rvs)] #TODO: why is this commented out?
          dt[, clin_rvs := split_to_vector(clin_rvs)] # Convert cleaned string to vector
        }

        # Return the cleaned data.table
        return(dt)
      }

      # Collapse and clean ICD and RVS columns
      # See above
      dt <- collapse_and_clean_icd_rvs(dt)

      # Helper function to clean and compare clinical columns
      clean_clinical_column <- function(col_name) {
        # Store the original column
        dt[, (paste0(col_name, "_orig")) := dt[[col_name]]]

        # Apply the clean_column function, which now returns a list of cleaned_col and is_covid
        cleaned_data <- clean_column(dt[[col_name]], na_like_strings, neoplasms_dt_actual)

        # Extract cleaned column and is_covid flag
        cleaned_col <- cleaned_data$cleaned_col
        is_covid_flag <- cleaned_data$is_covid

        # Update the cleaned column
        dt[, (col_name) := cleaned_col]

        # Store the original column as a string (for comparison purposes)
        dt[, (paste0(col_name, "_orig")) := sapply(get(paste0(col_name, "_orig")), toString)]

        # Convert the cleaned column to a string for comparison
        dt[, (col_name) := sapply(get(col_name), toString)]

        # Convert is_covid to logical (if it's currently a factor)
        dt[, is_covid := as.logical(as.character(is_covid))]

        # If is_covid is FALSE and is_covid_flag is TRUE, set is_covid to TRUE
        dt[, is_covid := ifelse(is_covid == FALSE & is_covid_flag == TRUE, TRUE, is_covid)]

        # Ensure is_covid is logical and replace any NA with FALSE
        dt[is.na(is_covid), is_covid := FALSE]

        # Compare cleaning results (only rows where the cleaned column differs from the original)
        comparison <- dt[
          !is.na(get(paste0(col_name, "_orig"))) & get(col_name) != get(paste0(col_name, "_orig")),
          .(
            old_code = get(paste0(col_name, "_orig")),
            new_code = get(col_name), count = .N
          ),
          by = .(get(paste0(col_name, "_orig")), get(col_name))
        ]

        return(comparison)
      }

      # Clean and compare c1 and c2 columns
      # See above
      c1_cleaning_comparison <- clean_clinical_column("c1")
      c2_cleaning_comparison <- clean_clinical_column("c2")

      # Function to replace multiple patterns with corresponding replacements
      replace_multiple_patterns <- function(text, patterns, replacements) {
        # Ensure patterns and replacements are the same length
        if (length(patterns) != length(replacements)) {
          stop("Patterns and replacements must have the same length.")
        }

        # Perform replacements
        modified_text <- stri_replace_all_regex(
          text,
          pattern = patterns,
          replacement = replacements,
          vectorize_all = FALSE # Apply all replacements simultaneously
        )

        # return the text after modification
        return(modified_text)
      }

      # Apply multi-replacement function to implement manual replacements
      dt[, clin_icd := lapply(clin_icd,
        replace_multiple_patterns,
        patterns = manual_patterns_to_replace,
        replacements = manual_code_replacements
      )]
      dt[, c1 := lapply(c1,
        replace_multiple_patterns,
        patterns = manual_patterns_to_replace,
        replacements = manual_code_replacements
      )]
      dt[, c2 := lapply(c2,
        replace_multiple_patterns,
        patterns = manual_patterns_to_replace,
        replacements = manual_code_replacements
      )]

      remove_lumped_icd_codes <- function(column) {
        ## Takes a column and separates out ICD-10 codes using "||"
        ## been lumped into a single string

        # Use regex to add "||" between letters and digits in the ICD codes (e.g., A123B456 -> A123||B456)
        modified_column <- stri_replace_all_regex(
          column, "(?<=\\d)(?=[A-Z]\\d{2,4})", "||",
          opts_regex = stri_opts_regex() # Specify regex options for the replacement
        )

        # Return the modified column with ICD codes split
        return(modified_column)
      }

      # See above
      # Handle any lumped ICD codes by splitting them
      dt[, c1 := remove_lumped_icd_codes(c1)]
      # Handle any lumped ICD codes by splitting them
      dt[, c2 := remove_lumped_icd_codes(c2)]

      split_to_vector <- function(column) {
        ## Splits a column of strings into vectors using "||" as the delimiter
        # column: the column to split

        result <- lapply(column, function(x) {
          # If the entry is NA, leave it as is
          if (is.na(x)) {
            return(NA_character_)
          } else {
            # Split the string into a vector using "||"
            return(unlist(strsplit(x, "||", fixed = TRUE)))
          }
        })

        # Return the list of vectors
        return(result)
      }

      # See above
      # Convert the cleaned columns into vectors
      dt[, c1 := split_to_vector(c1)]
      # Convert the cleaned columns into vectors
      dt[, c2 := split_to_vector(c2)]

      clean_clinical_columns <- function(dt) {
        ## Cleans and processes the clinical columns in a data.table
        # dt: input data.table with clinical columns

        # Deduplicate the ICD codes
        dt <- apply_add_c1_c2_to_clin_icd(dt)

        # TODO: append rvs to clin_proc, dont delete from c1 and c2
        # Process case rate 1 RVS codes
        c1_rvs_results <- append_and_remove_rvs(
          dt$clin_rvs, dt$c1, rvs_icd9
        )
        dt[, clin_rvs := c1_rvs_results$clin_rvs]
        dt[, c1 := c1_rvs_results$col]
        c1_discarded_rvs <- c1_rvs_results$discarded_rvs

        # Process case rate 2 RVS codes
        c2_rvs_results <- append_and_remove_rvs(
          dt$clin_rvs, dt$c2, rvs_icd9
        )
        dt[, clin_rvs := c2_rvs_results$clin_rvs]
        dt[, c2 := c2_rvs_results$col]
        c2_discarded_rvs <- c2_rvs_results$discarded_rvs

        # Ensure uniqueness of RVS codes in the final result
        # dt[, clin_rvs := lapply(clin_rvs, unique)]
        return(
          list(
            # Return the cleaned data.table
            dt = dt,
            # Return discarded RVS codes for checks
            discard_rvs_one = c1_discarded_rvs,
            discard_rvs_two = c2_discarded_rvs
          )
        )
      }
      # See above
      # Clean clinical columns
      clean_clin_col_res <- clean_clinical_columns(dt)
      dt <- clean_clin_col_res$dt

      # Replace empty strings with NA and remap patient data
      # See cleaning-functions.R
      replace_result <- replace_empty_with_na(dt = dt, to_view_checks)
      dt <- replace_result$return_data
      empty_strings_replaced_1 <- replace_result$return_replacement_summary

      remap_patient_data <- function(dt, to_view_checks = TRUE) {
        ## Remap the categorical columns in the inpatient data

        # Initialize unmapped variables and mapped data tables
        pat_unmap <- parent_unmap <- child_unmap <- discharge_unmap <- claim_status_unmap <- NULL
        pat_mapped <- parent_mapped <- child_mapped <- discharge_mapped <- claim_status_mapped <- NULL

        # Define the columns that need remapping
        columns_to_remap <- list(
          pat_type = "pat_type",
          pat_memcat_parent = "pat_memcat_parent",
          pat_memcat_child = "pat_memcat_child",
          clin_discharge = "clin_discharge",
          claim_status = "claim_status"
        )

        # Apply remapping for each column
        for (col_name in names(columns_to_remap)) {
          result <- remap_columns(dt, columns_to_remap[[col_name]], to_view_checks, known_values, remapped_column)

          # Update the original column using `set`
          set(dt, j = columns_to_remap[[col_name]], value = result$remapped)

          # Capture mapped and unmapped values
          if (col_name == "pat_type") {
            pat_mapped <- unique(data.table(Original = result$original, Mapped = result$remapped))
            if (length(result$unmapped) > 0 && to_view_checks) pat_unmap <- result$unmapped
          } else if (col_name == "pat_memcat_parent") {
            parent_mapped <- unique(data.table(Original = result$original, Mapped = result$remapped))
            if (length(result$unmapped) > 0 && to_view_checks) parent_unmap <- result$unmapped
          } else if (col_name == "pat_memcat_child") {
            child_mapped <- unique(data.table(Original = result$original, Mapped = result$remapped))
            if (length(result$unmapped) > 0 && to_view_checks) child_unmap <- result$unmapped
          } else if (col_name == "clin_discharge") {
            discharge_mapped <- unique(data.table(Original = result$original, Mapped = result$remapped))
            if (length(result$unmapped) > 0 && to_view_checks) discharge_unmap <- result$unmapped
          } else if (col_name == "claim_status") {
            claim_status_mapped <- unique(data.table(Original = result$original, Mapped = result$remapped))
            if (length(result$unmapped) > 0 && to_view_checks) claim_status_unmap <- result$unmapped
          }
        }

        # Return the remapped data and all mapping/unmapped data
        return(
          list(
            data = dt,
            pat_type_mapped = pat_mapped,
            pat_memcat_parent_mapped = parent_mapped,
            pat_memcat_child_mapped = child_mapped,
            clin_discharge_mapped = discharge_mapped,
            claim_status_mapped = claim_status_mapped,
            pat_type_unmapped = pat_unmap,
            memcat_parent_unmapped = parent_unmap,
            memcat_child_unmapped = child_unmap,
            discharge_unmapped = discharge_unmap,
            claim_status_unmapped = claim_status_unmap
          )
        )
      }

      # See above
      remapping_results <- remap_patient_data(dt, to_view_checks)

      dt <- remapping_results$data

      return(
        list(
          # data to return for further processing
          return_data = dt,
          # summary to return for checks and output
          return_summary = list(
            rename_success = rename_success,
            ICD_replacements_1 = c1_cleaning_comparison,
            ICD_replacements_2 = c2_cleaning_comparison,
            pat_type_mapped = remapping_results$pat_type_mapped,
            pat_memcat_parent_mapped = remapping_results$pat_memcat_parent_mapped,
            pat_memcat_child_mapped = remapping_results$pat_memcat_child_mapped,
            clin_discharge_mapped = remapping_results$clin_discharge_mapped,
            claim_status_mapped = remapping_results$claim_status_mapped,
            pat_type_unmapped = remapping_results$pat_type_unmapped,
            memcat_parent_unmapped = remapping_results$memcat_parent_unmapped,
            memcat_child_unmapped = remapping_results$memcat_child_unmapped,
            discharge_unmapped = remapping_results$discharge_unmapped,
            claim_status_unmapped = remapping_results$claim_status_unmapped,
            discard_rvs_one = clean_clin_col_res$discard_rvs_one,
            discard_rvs_two = clean_clin_col_res$discard_rvs_two,
            empty_strings_replaced_1 = empty_strings_replaced_1
          )
        )
      )
    }

    # Step 2: Clean the data in the 'chunk'
    # See above
    clean_result <- clean_data(chunk)
    chunk <- clean_result$return_data # Update chunk with cleaned data

    # Step 3: DEPRECATED

    # Define function to map RVS codes to ICD9 codes
    map_rvs_icd9 <- function(clin_rvs, rvs_icd9) {
      split_codes <- split_rvs_codes(rvs_icd9)
      rvs_maps <- create_rvs_map_lists(split_codes$with_drg)

      rvs_map_solo_env <- as.environment(rvs_maps$rvs_map_solo)

      return(
        list(
          # main return variable (a column) to save back to dt
          icd9_list = get_icd9_codes(clin_rvs, rvs_map_solo_env),
          # other return variables that are for checks and outputs
          rvs_map_list = rvs_maps$rvs_map_list,
          rvss = unique(unlist(clin_rvs)),
          mappable_rvs = intersect(unique(unlist(clin_rvs)), rvs_icd9$rvs),
          unmappable_rvs = setdiff(unique(unlist(clin_rvs)), rvs_icd9$rvs),
          multi_mapped_rvs = intersect(unique(unlist(clin_rvs)), names(rvs_maps$rvs_map_list)),
          without_drg = unique(rvs_icd9[!rvs %in% names(rvs_maps$rvs_map_list)]$rvs)
        )
      )
    }

    # Step 4: Map clinical RVS (Relative Value Scale) codes to ICD9 using 'rvs_icd9'
    # See above
    rvs_mapping_result <- map_rvs_icd9(chunk$clin_rvs, rvs_icd9)
    # Store the mapped ICD9 list into the chunk
    chunk[, icd9_list := rvs_mapping_result$icd9_list]

    # Step 5: DEPRECATED

    # Step 6: Extract columns c1, c2, and clin_icd for ICD10 mapping
    c1 <- chunk$c1
    c2 <- chunk$c2
    clin_icd <- chunk$clin_icd

    implement_icd10_mapping <- function(c1, c2, clin_icd, tdrg_icd10) {
      # Step 1: Get all unique ICD codes from the provided columns (c1, c2, and clin_icd)
      icds <- get_unique_icd_codes(c1, c2, clin_icd)

      # Step 2: Create an environment for Thai ICD10 codes for faster lookup
      # This uses the unique set of ICD10 codes in the tdrg_icd10 table.
      thai_icd10_env <- create_thai_icd10_environment(
        unique(tdrg_icd10$CODE)
      )

      # Step 3: Create a second environment for Thai ICD10 neoplasm codes (those with slashes '/')
      neoplasms_env <- create_thai_icd10_environment(
        unique(tdrg_icd10[grepl("/", tdrg_icd10$CODE), "CODE"])
      )

      # Step 4: Find direct matches between the provided ICD codes (icds) and the Thai ICD10 environment
      direct_match_codes <- find_direct_icd_matches(
        icds, thai_icd10_env
      )

      # Step 5: Generate the full ICD10 mapping for the ICD codes,
      # considering both Thai ICD10 environment and neoplasms environment.
      icd_mapping_info <- generate_icd10_mapping(
        icds, thai_icd10_env, neoplasms_env, covid_rvs
      )
      # Extract the mapping and the count of modified mappings
      icd_mapping <- icd_mapping_info$icd_mapping_res

      modified_count <- icd_mapping_info$modified_count

      # Step 6: Identify ICD codes that were not successfully mapped.
      unmatched_icds <- setdiff(icds, names(icd_mapping))

      # Step 7: If there are unmatched ICD codes, gather their source information (c1, c2, clin_icd)
      # and the count of occurrences in each column.
      if (length(unmatched_icds) > 0) {
        unmatched_sources <- data.table(
          code = unmatched_icds, source = NA_character_, count = 0
        )
        # Loop over the columns (c1, c2, clin_icd) to fill in source and count details for unmatched codes.
        for (col_name in c("c1", "c2", "clin_icd")) {
          col_values <- get(col_name)
          unmatched_sources[
            code %in% unlist(col_values),
            source := col_name
          ]
          unmatched_sources[
            code %in% unlist(col_values),
            count := count + table(unlist(col_values))[code]
          ]
        }
        # Order unmatched codes by their occurrence count in descending order
        unmatched_sources <- unmatched_sources[order(-count)]
      } else {
        # If there are no unmatched codes, return an empty data.table.
        unmatched_sources <- data.table()
      }

      # Step 8: Create a data.table containing the mapping between PHL (input) ICD10 codes
      # and Thai DRG ICD10 codes.
      icd10_map <- data.table(
        phl_icd10 = names(icd_mapping),
        tdrg_icd10 = unlist(icd_mapping)
      )

      # Step 9: DEPRECATED

      # Step 10: Create an environment from the ICD10 mapping for fast lookup during column mapping.
      icd10_env <- list2env(
        setNames(as.list(icd10_map$tdrg_icd10), icd10_map$phl_icd10)
      )

      # Step 11: Apply the ICD10 mapping to the columns c1, c2, and clin_icd
      # This updates these columns based on the generated ICD10 environment.
      mapped_columns <- apply_icd10_mapping_to_columns(
        c1, c2, clin_icd, icd10_env
      )

      # Step 12: Return a list containing the mapped columns and other information for further checks and outputs:
      # - The updated columns (c1, c2, clin_icd)
      # - The full ICD10 map (icd10_map_dt)
      # - The unique ICD codes
      # - Direct matches found
      # - Unmatched ICDs and their source information
      return(
        list(
          c1 = mapped_columns$c1,
          c2 = mapped_columns$c2,
          clin_icd = mapped_columns$clin_icd,
          icd10_map_dt = icd10_map,
          unique_icds = icds,
          direct_matches = direct_match_codes,
          unmatched = unmatched_icds,
          unmatched_sources = unmatched_sources,
          icd_mapping_res = icd_mapping
        )
      )
    }

    # Step 7: Perform ICD10 mapping using the extracted columns and 'tdrg_icd10' mapping data
    # See above
    icd10_mapping_result <- implement_icd10_mapping(
      c1, c2, clin_icd, tdrg_icd10
    )

    # Update chunk with the mapped ICD10 codes
    chunk[, c1 := icd10_mapping_result$c1]
    chunk[, c2 := icd10_mapping_result$c2]
    chunk[, clin_icd := icd10_mapping_result$clin_icd]

    # Step 8: Replace any empty strings with NA values, returning a summary of replacements
    # See cleaning-functions.R
    res2 <- replace_empty_with_na(dt = chunk, to_view_checks)
    chunk <- res2$return_data # Update chunk with cleaned data
    empty_strings_replaced_2 <- res2$return_replacement_summary # Store replacement summary

    # Step 9: Define a function to remove all whitespace from character vectors
    remove_whitespace <- function(x) {
      if (is.null(x) || length(x) == 0) {
        return(NA_character_) # Return NA for NULL or empty lists
      } else {
        return(gsub("\\s+", "", x)) # Remove all whitespace characters
      }
    }

    # Step 10: Apply the remove_whitespace function to the list columns 'c1', 'c2', and 'clin_icd'
    # See above
    chunk[, c1 := lapply(c1, remove_whitespace)]
    chunk[, c2 := lapply(c2, remove_whitespace)]
    chunk[, clin_icd := lapply(clin_icd, remove_whitespace)]

    # Step 11: DEPRECATED

    apply_find_pdx <- function(c1, c2, clin_icd, acc_pdx) {
      ## Function to apply the PDX finding logic in a vectorized manner

      # Step 1: Create a new environment for accepted PDX codes
      acc_pdx_env <<- new.env(hash = TRUE, parent = emptyenv())

      # Step 2: Populate the environment with accepted PDX codes
      for (code in acc_pdx) {
        assign(code, TRUE, envir = acc_pdx_env)
      }

      # Define helper function to find PDX for each row
      find_pdx_for_row <- function(c1, c2, clin_icd) {
        # Split c1 and c2 by '|' if necessary
        c1 <- unlist(strsplit(c1, "\\|"))
        c2 <- unlist(strsplit(c2, "\\|"))

        # Step 1: Check if any element in c1 or c2 is an acceptable PDx
        for (cr_list in list(c1, c2)) {
          for (cr in cr_list) {
            if (!is.na(cr) && exists(cr, envir = acc_pdx_env)) {
              return(list(pdx = cr, pdx_code = ifelse(cr %in% c1, 1, 2)))
            }
          }
        }

        # Step 2: Unlist clin_icd by splitting if necessary
        clin_icd <- unlist(strsplit(clin_icd, "\\|"))

        # Step 3: Get a list of acceptable PDx from clin_icd
        pdxs <- unique(clin_icd)
        pdxs <- pdxs[sapply(pdxs, function(x) exists(x, envir = acc_pdx_env))]

        # Step 4: Handle cases with no or only one acceptable PDx
        if (length(pdxs) == 0) {
          return(list(pdx = NA_character_, pdx_code = 99))
        } else if (length(pdxs) == 1) {
          return(list(pdx = pdxs[1], pdx_code = 3))
        }

        # Function to check similarity between two strings
        check_similarity <- function(x, y) {
          score <- 0
          min_len <- min(nchar(x), nchar(y))
          for (i in 1:min_len) {
            if (substr(x, i, i) == substr(y, i, i)) {
              score <- score + 1
            }
          }
          return(score)
        }

        # Step 5: Check c1 and c2 for matching starting letters
        for (cr_list in list(c1, c2)) {
          for (cr in cr_list) {
            if (!is.na(cr)) {
              starting_letter <- substr(cr, 1, 1)
              starting_codes <- pdxs[substr(pdxs, 1, 1) == starting_letter]

              if (length(starting_codes) == 1) {
                return(list(pdx = starting_codes[1], pdx_code = 4))
              } else if (length(starting_codes) > 1) {
                starting_codes <- starting_codes[
                  # See above
                  order(sapply(starting_codes, function(x) check_similarity(cr, x)), decreasing = TRUE)
                ]
                return(list(pdx = starting_codes[1], pdx_code = 5))
              }
            }
          }
        }

        # Step 6: If no matching starting letter, pick a random PDx
        if (length(pdxs) > 0) {
          return(list(pdx = sample(pdxs, 1), pdx_code = 6))
        }

        # Step 7: Return NA and code 99 if no PDx is found
        return(list(pdx = NA_character_, pdx_code = 99))
      }

      # Step 3: Apply find_pdx_for_row function to all rows
      # See above
      result <- mapply(find_pdx_for_row, c1, c2, clin_icd, SIMPLIFY = FALSE)

      # Step 4: Extract PDX and PDX codes into vectors
      pdx <- sapply(result, function(x) x$pdx)
      pdx_code <- sapply(result, function(x) x$pdx_code)

      # Return the PDX values and codes
      return(list(pdx = pdx, pdx_code = pdx_code))
    }

    # Step 12: Apply a function to find the primary diagnosis (pdx) based on 'c1', 'c2', and 'clin_icd'
    # See above
    pdx_result <- apply_find_pdx(
      chunk$c1, chunk$c2, chunk$clin_icd, acc_pdx
    )
    # Store the primary diagnosis (pdx) and its code into the chunk
    chunk$pdx <- pdx_result$pdx
    chunk$pdx_code <- pdx_result$pdx_code

    # Step 13: DEPRECATED

    # Step 14: Define a function to remove the primary diagnosis (pdx) from list columns (c1, c2, clin_icd)
    remove_pdx_from_list <- function(pdx, lst) {
      if (!is.na(pdx)) {
        # Remove the primary diagnosis from the list
        lst <- setdiff(lst, pdx)
      }
      return(lst)
    }

    # Step 15: Apply the 'remove_pdx_from_list' function to each row of 'c1', 'c2', and 'clin_icd'
    chunk[, c1 := lapply(seq_len(.N), function(i) as.character(remove_pdx_from_list(pdx[i], c1[[i]])))]
    chunk[, c2 := lapply(seq_len(.N), function(i) as.character(remove_pdx_from_list(pdx[i], c2[[i]])))]
    chunk[, clin_icd := lapply(seq_len(.N), function(i) as.character(remove_pdx_from_list(pdx[i], clin_icd[[i]])))]

    # Step 16: Create a summary by combining clean results and ICD10 mapping information
    chunk_summary <- modifyList(
      clean_result$return_summary,
      list(
        unique_icds = icd10_mapping_result$unique_icds,
        direct_matches = icd10_mapping_result$direct_matches,
        unmatched = icd10_mapping_result$unmatched,
        unmatched_sources = icd10_mapping_result$unmatched_sources,
        icd10_map_dt = icd10_mapping_result$icd10_map_dt,
        rvss = rvs_mapping_result$rvss,
        mappable_rvs = rvs_mapping_result$mappable_rvs,
        unmappable_rvs = rvs_mapping_result$unmappable_rvs,
        multi_mapped_rvs = rvs_mapping_result$multi_mapped_rvs,
        without_drg = rvs_mapping_result$without_drg
      )
    )

    # Step 17: Optionally trigger garbage collection to reduce memory usage
    gc()

    # Step 18: Return the processed chunk and summary information
    return(
      list(
        return_chunk = chunk, # Return the processed chunk data
        return_summary = chunk_summary # Return the summary for checks and outputs
      )
    )
  }

  # Step 9: Apply parallel processing (Unix uses 'mclapply', non-Unix uses 'future_lapply')
  if (to_parallel) {
    if (to_debug) message("Conducting mclapply")
    parallel_results <- mclapply(
      chunks, process_chunk,
      mc.cores = nthreads
    )
  } else {
    if (to_debug) message("Conducting lapply")
    parallel_results <- lapply(
      chunks, process_chunk
    )
  }

  # Step 10: Combine results from all parallel chunks
  parallel_summaries <- lapply(parallel_results, function(res) res$return_summary)
  rbound_dt <- rbindlist(lapply(parallel_results, function(res) res$return_chunk))

  combined_chunk_summary <- combine_chunk_summaries(
    parallel_summaries, tmp_nrow
  )

  # Step 11: Check for invalid primary diagnoses (PDx) and update the summary
  acc_pdx_env <- new.env(hash = TRUE, parent = emptyenv())
  for (code in acc_pdx) {
    assign(code, TRUE, envir = acc_pdx_env)
  }
  invalid_pdx_indices <- which(
    !is.na(rbound_dt$pdx) & rbound_dt$pdx != "" &
      !sapply(rbound_dt$pdx, function(x) exists(x, acc_pdx_env))
  )
  if (length(invalid_pdx_indices) > 0) {
    message(paste("Invalid PDx found:", rbound_dt$pdx[invalid_pdx_indices]))
    combined_chunk_summary$pdx_success <- FALSE
  } else {
    combined_chunk_summary$pdx_success <- TRUE
  }

  gc()

  summarized_dt <- rbound_dt # Store the summarized data
  combined_parallel_summary <- combined_chunk_summary # Store combined summary
  combined_parallel_summary$replacement_summary <- read_in_replacement_summary

  # Step 12: Write processed data to checkpoint file if required
  if (to_write) {
    saveRDS(
      summarized_dt, here(checkpoint_1_path, paste0(
        checkpoint_1_prefix, year_to_load, suffix,
        "part_", sprintf("%02d", loop_part), "_of_", split_parts, ".rds"
      )),
      compress = TRUE
    )
  }

  # Step 13: Collect summaries for each part
  all_parts_summaries[[loop_part]] <- combined_parallel_summary
  processing_times[[loop_part]] <- as.numeric(difftime(Sys.time(), start_time, units = "secs"))

  # Step 14: Update status and ETA
  print_status_update(loop_part, split_parts, processing_times, "clean")

  if (loop_part == 1) dim_dt <<- dim(summarized_dt)
  nrow_end[[loop_part]] <<- nrow(summarized_dt)

  # Step 15: Clean up memory after processing each part
  rm(read_in_dt, rbound_dt, summarized_dt)
  gc()
}


In [ ]:
# Step 16: Ensure that row counts match between parts
for (nrow_part in 1:split_parts) {
  if (nrow_start[[nrow_part]] != nrow_end[[nrow_part]]) {
    warning(
      "WARNING: Row Count Mismatch! Part ", nrow_part,
      " has ", nrow_start[[nrow_part]], " starting rows and ",
      nrow_end[[nrow_part]], " ending rows\n"
    )
    stop("ERROR: Row Count Mismatch")
  }
}
message("\nRow Counts Match for All Parts\n")

# Step 17: Combine all parts into a master data table
for (read_part in 1:split_parts) {
  master_dt_list[[read_part]] <- readRDS(here(checkpoint_1_path, paste0(
    checkpoint_1_prefix, year_to_load, suffix,
    "part_", sprintf("%02d", read_part), "_of_", split_parts, ".rds"
  )))
}
master_dt <<- rbindlist(master_dt_list)
rm(master_dt_list)
gc()
if (to_write) {
  saveRDS(master_dt, here(checkpoint_2_path, paste0(
    checkpoint_2_prefix, year_to_load, suffix, ".rds"
  )), compress = TRUE)
}

# Step 18: Print final summaries
print_summary_tables(
  combine_parts_summaries(all_parts_summaries, tmp_nrow),
  end_nrow
)


In [ ]:
if (exists("master_dt")) {
  result <- data.table::copy(master_dt)
  rm(master_dt)
  gc()
} else {
  result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, ".rds")))
  gc()
}

# Filter rows where any element in c1 contains "c("
if (to_debug) filtered_rows <- as.data.table(result[sapply(c1, function(x) any(grepl("c\\(", x)))])

# Display the structure of the filtered rows
if (to_debug) str(filtered_rows)

# Track invalid age corrections
invalid_age_before <- nrow(result[pat_age < -1 | pat_age > 124, .(id_series)])

pat_age_less_than_or_equal_to_neg_one <- result[pat_age <= -1, .(id_series, pat_age, c1, c2)]
fwrite(pat_age_less_than_or_equal_to_neg_one, here(debug_path, "pat_age_less_than_or_equal_to_neg_one.csv"))

pat_age_between_zero_and_neg_one <- result[pat_age < 0 & pat_age > -1, .(id_series, pat_age, c1, c2)]
fwrite(pat_age_between_zero_and_neg_one, here(debug_path, "pat_age_between_zero_and_neg_one.csv"))

# Use c1_orig and c2_orig as clin_c1 and clin_c2
result[, clin_c1 := c1_orig]
result[, clin_c2 := c2_orig]
result[, c("c1_orig", "c2_orig") := NULL]

# Use icd9_list as clin_proc
result[, clin_proc := icd9_list]
result[, icd9_list := NULL]

# Remove the primary diagnosis from the list of secondary diagnoses
result[, clin_icd := Map(function(pdx_var, sdx_var) sdx_var[sdx_var != pdx_var], pdx, clin_icd)]
result[, clin_sdx := clin_icd]
result[, clin_icd := NULL]

# Ensure that NA check respects the structure of c1 and returns a logical vector of the same length as result
if (to_debug) str(result[sapply(c1, function(x) length(x) > 1 && !all(is.na(unlist(x))))])

if (!"pat_bdate" %in% colnames(result)) {
  if (to_debug) print(colnames(result))
  result[, pat_bdate := NA_Date_]
}

if (to_debug) print(unique(result$pat_bdate))

date_cols <- c("date_adm", "date_dis", "date_rec", "date_ref", "date_check", "pat_bdate", "date_ext")
# Function to convert and replace dates before 1900-01-01 with NA
convert_and_filter_dates <- function(x) {
  converted_dates <- as.Date(x, format = "%m/%d/%Y")
  # Replace dates before 1900-01-01 with NA
  converted_dates[converted_dates < as.Date("1900-01-01")] <- NA_Date_
  return(converted_dates)
}

if (is_unix) {
  result[, (date_cols) := mclapply(.SD, convert_and_filter_dates, mc.cores = parallel::detectCores()), .SDcols = date_cols]
} else {
  result[, (date_cols) := lapply(.SD, convert_and_filter_dates), .SDcols = date_cols]
}

# Process time columns
time_cols <- c("time_adm", "time_dis")
standardize_time <- function(x) {
  x <- ifelse(is.na(x), "00:00:00", paste0(x, ":00"))
  as.ITime(x)
}
if (is_unix) {
  result[, (time_cols) := mclapply(.SD, standardize_time, mc.cores = parallel::detectCores()), .SDcols = time_cols]
} else {
  result[, (time_cols) := lapply(.SD, standardize_time), .SDcols = time_cols]
}

# Convert ITime object to character in HH:MM:SS format
result[, time_adm := strftime(time_adm, format = "%H:%M:%S")]
result[, time_dis := strftime(time_dis, format = "%H:%M:%S")]

# Convert date_adm and date_dis from Asia/Manila to UTC
result[, date_adm := as.POSIXct(paste(date_adm, time_adm), format = "%Y-%m-%d %H:%M:%S", tz = "Asia/Manila")]
result[, date_dis := as.POSIXct(paste(date_dis, time_dis), format = "%Y-%m-%d %H:%M:%S", tz = "Asia/Manila")]

# Process logical columns
result[, clin_outpatient := as.logical(as.integer(clin_outpatient))]
result[, clin_emergency := as.logical(as.integer(clin_emergency))]

# Process numeric columns
if (!"pat_bwt" %in% colnames(dt)) {
  result[, pat_bwt := NA_real_]
}
num_cols <- c("pat_age", "pat_bwt", "clin_discharge", "claim_payout", "claim_charge", "id_year", "pdx_code")
if (is_unix) {
  result[, (num_cols) := mclapply(.SD, as.numeric, mc.cores = parallel::detectCores()), .SDcols = num_cols]
} else {
  result[, (num_cols) := lapply(.SD, as.numeric), .SDcols = num_cols]
}

# Process integer columns
int_cols <- c("clin_discharge", "id_year", "pdx_code")
if (is_unix) {
  result[, (int_cols) := mclapply(.SD, as.integer, mc.cores = parallel::detectCores()), .SDcols = int_cols]
} else {
  result[, (int_cols) := lapply(.SD, as.integer), .SDcols = int_cols]
}

# Process character columns
char_cols <- c("id_hcp", "pat_type", "clin_acc", "pat_rel", "pat_sex", "pat_memcat_parent", "pat_memcat_child", "claim_status", "pdx")
if (is_unix) {
  result[, (char_cols) := mclapply(.SD, as.character, mc.cores = parallel::detectCores()), .SDcols = char_cols]
} else {
  result[, (char_cols) := lapply(.SD, as.character), .SDcols = char_cols]
}
# initialize age
result[, pat_ageday := NA_integer_]

# START OF AGE AND BDAY CORRECTION
# Age correction logic
invalid_ages_before_correction <- result[pat_age < 0 | pat_age > 124, .N]
invalid_age_ids_before <- result[pat_age < 0 | pat_age > 124, id_series]


# Step 1: Fix pat_age for specific ranges
result[!is.na(pat_age) & pat_age > 0, pat_age := floor(pat_age)]
result[pat_age < 0 & pat_age >= -1, pat_age := 0] # Set ages between -1 and 0 to 0
result[pat_age < -1 | pat_age > 124, pat_age := NA_integer_] # Set pat_age to NA if greater than 124 or less than -1

# Step 2: Recalculate pat_age only if necessary
# Subset the rows that meet the condition before recalculation
recalculated_rows <- result[
  !is.na(pat_bdate) & !is.na(pat_age) &
    pat_age != floor(as.numeric(seconds(as.Date(date_adm) - pat_bdate)) / 365.25)
]

# Print the rows where recalculation is going to happen (before recalculation)
cat("Rows where pat_age is being recalculated (Before):\n")
print(recalculated_rows[, .(pat_bdate, date_adm, pat_age)])

# Perform the recalculation and store the new values in a separate column for comparison
result[
  !is.na(pat_bdate) & !is.na(pat_age) &
    pat_age != floor(as.numeric(seconds(as.Date(date_adm) - pat_bdate)) / 365.25),
  pat_age_recalculated := floor(as.numeric(seconds(as.Date(date_adm) - pat_bdate)) / 365.25)
]

# Show before and after recalculated pat_age
cat("Before and After Recalculation:\n")
print(result[!is.na(pat_age_recalculated), .(pat_bdate, date_adm, pat_age, pat_age_recalculated)])

# Save pat_age_recalculated to pat_age, then delete pat_age_recalculated
result[!is.na(pat_age_recalculated) & pat_age_recalculated > 0, pat_age := pat_age_recalculated]
result[, pat_age_recalculated := NULL] # Remove the recalculated column

# Regenerate or correct DOB
invalid_bdate_before <- result[is.na(pat_bdate), .N]
invalid_bdate_ids_before <- result[is.na(pat_bdate), id_series]

# result[!is.na(pat_age) & is.na(pat_bdate), pat_bdate := dmy(generate_dob(format(pat_bdate, "%Y-%m-%d"), pat_age, format(date_adm, "%Y-%m-%d")))]
# result[!is.na(pat_bdate) & pat_bdate <= date_adm, pat_age := floor(as.numeric(interval(pat_bdate, date_adm) / years(1)))]

# Save invalid age rows to CSV
invalid_age_path <- here("data-cleaning", "debug", "invalid_age.csv")
fwrite(data.table(id_series = invalid_age_ids_before), invalid_age_path)

# Save invalid birthdate rows to CSV
invalid_bdate_path <- here("data-cleaning", "debug", "invalid_bdate.csv")
fwrite(data.table(id_series = invalid_bdate_ids_before), invalid_bdate_path)

# Print messages for invalid ages corrected
invalid_ages_after_correction <- result[pat_age < 0 | pat_age > 124, .N]
message(
  "Number of invalid ages corrected: ", invalid_ages_before_correction - invalid_ages_after_correction,
  ". Invalid ages are those with a value less than 0 or greater than 124, which were reset to NA or corrected."
)
# END OF AGE AND BDAY CORRECTION

# Assuming acc_icd_env is an environment containing acc_icd codes
acc_icd_env <- new.env(hash = TRUE, parent = emptyenv())
for (code in acc_icd) {
  assign(code, TRUE, envir = acc_icd_env)
}

# Modify the data.table operation to use mget with the acc_icd_env
result[, clin_sdx := lapply(clin_sdx, function(row) {
  codes <- unlist(row)
  valid_codes <- codes[!is.na(mget(codes, envir = acc_icd_env, ifnotfound = NA_character_))]
  if (length(valid_codes) > 0) {
    return(valid_codes)
  } else {
    return(NA_character_)
  }
})]

# Optionally unlist each element of clin_sdx
result[, clin_sdx := lapply(clin_sdx, unlist)]

na_replaced_result <- replace_empty_with_na(result)

result <- na_replaced_result$return_data

# Process character columns and convert to UTF-8
if (is_unix) {
  result[, (char_cols) := mclapply(.SD, function(col) iconv(col, from = "", to = "UTF-8"), mc.cores = parallel::detectCores()), .SDcols = char_cols]
} else {
  result[, (char_cols) := lapply(.SD, function(col) iconv(col, from = "", to = "UTF-8")), .SDcols = char_cols]
}

# Function to clean data by handling missing values and replacing invalid entries
clean_data_columns <- function(data) {
  # Identify and process character columns
  char_cols <- names(data)[sapply(data, is.character)]

  # If running on Unix, apply parallel processing for character columns
  if (is_unix) {
    data[, (char_cols) := mclapply(.SD, function(col) {
      # Replace "None" and empty strings with NA in character columns
      col[col %in% c("None", "")] <- NA_character_
      return(col) # Return modified column
    }, mc.cores = parallel::detectCores()), .SDcols = char_cols]
  } else {
    # Apply sequential processing for character columns (non-Unix systems)
    data[, (char_cols) := lapply(.SD, function(col) {
      col[col %in% c("None", "")] <- NA_character_
      return(col) # Return modified column
    }), .SDcols = char_cols]
  }

  # Identify and process numeric columns
  num_cols <- names(data)[sapply(data, is.numeric)]

  # If running on Unix, apply parallel processing for numeric columns
  if (is_unix) {
    data[, (num_cols) := mclapply(.SD, function(col) {
      # Replace NaN values with NA in numeric columns
      col[is.nan(col)] <- NA_real_
      return(col) # Return modified column
    }, mc.cores = parallel::detectCores()), .SDcols = num_cols]
  } else {
    # Apply sequential processing for numeric columns (non-Unix systems)
    data[, (num_cols) := lapply(.SD, function(col) {
      col[is.nan(col)] <- NA_real_
      return(col) # Return modified column
    }), .SDcols = num_cols]
  }

  # Identify and process list columns
  list_cols <- names(data)[sapply(data, is.list)]

  # If running on Unix, apply parallel processing for list columns
  if (is_unix) {
    data[, (list_cols) := mclapply(.SD, function(col) {
      # For each list element, check if it contains characters and replace "None" and empty strings with NA
      lapply(col, function(x) {
        if (is.character(x)) x[x %in% c("None", "")] <- NA_character_
        return(x) # Return modified list element
      })
    }, mc.cores = parallel::detectCores()), .SDcols = list_cols]
  } else {
    # Apply sequential processing for list columns (non-Unix systems)
    data[, (list_cols) := lapply(.SD, function(col) {
      lapply(col, function(x) {
        if (is.character(x)) x[x %in% c("None", "")] <- NA_character_
        return(x) # Return modified list element
      })
    }), .SDcols = list_cols]
  }

  return(data) # Return the cleaned data.table
}

# Apply the cleaning function to the result data.table
result <- clean_data_columns(result)

# Convert string columns to arrays, handling different delimiters: comma, comma with space, single pipe, and double pipe
array_columns <- c("id_hcp")
split_pattern <- "\\s*,\\s*|\\|\\||\\|" # Regex pattern to handle commas, single pipes, and double pipes

if (is_unix) {
  result[, (array_columns) := mclapply(.SD, function(x) {
    # Split based on the specified pattern (comma, comma with space, single pipe, or double pipe)
    x <- strsplit(x, split_pattern)
    # Handle empty or NA entries
    lapply(x, function(y) if (length(y) == 0L || all(is.na(y))) character(0) else y)
  }, mc.cores = parallel::detectCores()), .SDcols = array_columns]
} else {
  result[, (array_columns) := lapply(.SD, function(x) {
    # Split based on the specified pattern (comma, comma with space, single pipe, or double pipe)
    x <- strsplit(x, split_pattern)
    # Handle empty or NA entries
    lapply(x, function(y) if (length(y) == 0L || all(is.na(y))) character(0) else y)
  }), .SDcols = array_columns]
}

# Ensure 'clin_sdx', 'clin_proc', and 'id_hcp' are not NULL
list_columns <- c("clin_sdx", "clin_proc", "id_hcp")
if (is_unix) {
  result[, (list_columns) := mclapply(.SD, function(col) {
    lapply(col, function(x) if (is.null(x) || length(x) == 0L || all(is.na(x))) character(0) else x)
  }, mc.cores = parallel::detectCores()), .SDcols = list_columns]
} else {
  result[, (list_columns) := lapply(.SD, function(col) {
    lapply(col, function(x) if (is.null(x) || length(x) == 0L || all(is.na(x))) character(0) else x)
  }), .SDcols = list_columns]
}

setnames(result, c("pdx", "pdx_code"), c("clin_pdx", "clin_pdx_source"))

setcolorder(result, c(
  "id_year", "id_series", "id_pin", "id_hci", "id_hcp", "date_adm", "time_adm", "date_dis", "time_dis",
  "date_rec", "date_ref", "date_check", "date_ext", "pat_type", "pat_rel", "pat_bdate", "pat_age",
  "pat_ageday", "pat_sex", "pat_bwt", "pat_memcat_parent", "pat_memcat_child", "is_covid", "claim_status", "claim_payout",
  "claim_charge", "clin_discharge", "clin_outpatient", "clin_emergency", "clin_acc", "clin_c1", "c1", "clin_c2", "c2",
  "clin_sdx", "clin_proc", "clin_rvs", "clin_pdx", "clin_pdx_source"
))

# Use a temporary column to avoid self-reference
result[, temp_clin_discharge := as.integer(clin_discharge)]

# Assign the temp column back to clin_discharge
result[, clin_discharge := temp_clin_discharge]

# Remove the temporary column
result[, temp_clin_discharge := NULL]

saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")), compress = TRUE)


In [ ]:
# Convert both id_hci and PMCC_NO to characters to ensure consistency for joining
result[, id_hci := as.character(id_hci)]
hci[, PMCC_NO := as.character(PMCC_NO)]

# Remove leading zeros from numeric PMCC_NO values, while preserving non-numeric values
hci[, PMCC_NO_stripped := fifelse(
  grepl("^[0-9]+$", PMCC_NO),
  sub("^0+", "", PMCC_NO), # Remove leading zeros from numeric strings
  PMCC_NO # Keep non-numeric values unchanged
)]

# Keep only necessary columns from hci for the join
hci_subset <- hci[, .(PMCC_NO_stripped, SOC_SECTOR, INST_NAME, CAT_24, REGION_NAME, PROVINCE_NAME)]

# Perform a left join, keeping all rows in result and only matching rows from hci
result <- merge(
  result,
  hci_subset,
  by.x = "id_hci",
  by.y = "PMCC_NO_stripped",
  all.x = TRUE, # Keep all rows from result
  all.y = FALSE # Only include matching rows from hci
)

# ----- Filter rows by allowed CAT_24 categories -----
allowed_categories <- c("INFIRMARY/DISPENSARY", "LEVEL 1 HOSPITAL", "LEVEL 2 HOSPITAL", "LEVEL 3 HOSPITAL")

# Count rows where CAT_24 is not in allowed categories
count_excluded <- result[!(CAT_24 %in% allowed_categories), .N]
cat("Number of rows where CAT_24 is not in the allowed categories:", count_excluded, "\n")

# Print rows to be excluded
if (to_debug) cat("Rows to be dropped due to invalid CAT_24:\n")
if (to_debug) print(result[!(CAT_24 %in% allowed_categories)])

# Drop rows where CAT_24 is not in the allowed categories
result <- result[CAT_24 %in% allowed_categories]
if (to_debug) str(result)
# ----- Filter rows where clin_outpatient is TRUE -----
# Count rows where clin_outpatient is TRUE
count_clin_outpatient_true <- result[clin_outpatient == TRUE, .N]
cat("Number of rows where clin_outpatient is TRUE:", count_clin_outpatient_true, "\n")

# Print rows to be excluded
if (to_debug) cat("Rows to be dropped due to clin_outpatient being TRUE:\n")
if (to_debug) print(result[clin_outpatient == TRUE])

# Drop rows where clin_outpatient is TRUE
result <- result[clin_outpatient != TRUE]
if (to_debug) str(result)
# ----- Filter rows where claim_status is not "G" -----
# Count rows where claim_status is not "G"
count_claim_status_not_G <- result[claim_status != "G", .N]
cat("Number of rows where claim_status is not 'G':", count_claim_status_not_G, "\n")

# Print rows to be excluded
if (to_debug) cat("Rows to be dropped due to claim_status not being 'G':\n")
if (to_debug) print(result[claim_status != "G"])

# Drop rows where claim_status is not "G"
result <- result[claim_status == "G"]
if (to_debug) str(result)
# Convert 'c1' from a list of character vectors into a single concatenated string
result[, c1_spc := sapply(c1, function(x) {
  if (is.null(x) || all(is.na(x))) {
    return(NA_character_) # Return NA if the list is empty or all values are NA
  } else {
    return(paste(sort(unique(x)), collapse = ",")) # Sort, remove duplicates, and concatenate
  }
})]

# ----- Handle duplicate rows based on specific columns -----
duplicate_columns <- c("id_pin", "pat_type", "pat_age", "pat_sex", "date_adm", "date_dis", "c1_spc", "claim_payout")

# Count duplicate rows based on the specified columns
count_duplicates <- result[duplicated(result[, ..duplicate_columns]), .N]
cat("Number of duplicated rows based on specified columns:", count_duplicates, "\n")

# Print duplicate rows
if (to_debug) cat("Rows with duplicated combinations of the specified columns:\n")
if (to_debug) print(result[duplicated(result[, ..duplicate_columns])])

# Drop duplicate rows, keeping only the first occurrence
result <- result[!duplicated(result[, ..duplicate_columns])]
if (to_debug) str(result)
# ----- Remove unnecessary columns and reorder remaining columns -----
# Remove unnecessary columns
result[, c("c1", "c1_spc", "c2", "clin_rvs") := NULL]

# Perform garbage collection to free up memory
gc()

# Set the column order to a specified structure
setcolorder(result, c(
  "id_year", "id_series", "id_pin", "id_hci", "id_hcp", "date_adm", "time_adm", "date_dis", "time_dis",
  "date_rec", "date_ref", "date_check", "date_ext", "pat_type", "pat_rel", "pat_bdate", "pat_age",
  "pat_ageday", "pat_sex", "pat_bwt", "pat_memcat_parent", "pat_memcat_child", "is_covid", "claim_status", "claim_payout",
  "claim_charge", "clin_discharge", "clin_outpatient", "clin_emergency", "clin_acc", "clin_c1", "clin_c2",
  "clin_sdx", "clin_proc", "clin_pdx", "clin_pdx_source", "SOC_SECTOR", "CAT_24", "INST_NAME", "REGION_NAME", "PROVINCE_NAME"
))

saveRDS(result, here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "stata", ".rds")), compress = TRUE)


In [ ]:
# str(result_after_cleaning)
print(result[grepl("e", id_series)])
print(result[grepl("e", id_pin)])
print(result[grepl("e", id_hci)])
# str(result)
# fwrite(result, "test.csv")
# Search for rows where any element in clin_sdx is "A"
result[, if (any(sapply(clin_sdx, function(row) "A" %in% row))) print(.SD), by = 1:nrow(result)]


In [ ]:
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))
date_cols <- c("date_adm", "date_dis", "date_rec", "date_ref", "date_check", "pat_bdate", "date_ext")

# Find rows where any date column has a date before 1900-01-01
rows_with_old_dates <- result[Reduce(`|`, lapply(.SD, function(x) x < as.Date("1900-01-01"))), .SDcols = date_cols]

# Print the resulting rows
print(rows_with_old_dates)


In [ ]:
result <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))
result[, c("c1", "c2", "clin_rvs") := NULL]
# cat(unique(result$id_series), sep = "\n")
# test <- fread("/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/claims/raw/claims_extract_CLAIMS 2019.csv", nrows = 1000, colClasses = "character")
# print(head(trimws(unique(test$PSEUDO_CLAIMSERIES))))


In [ ]:
before <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, ".rds")))
after <- readRDS(here(checkpoint_2_path, paste0(checkpoint_2_prefix, year_to_load, suffix, "final", ".rds")))

# Ensure both data.tables have the same key columns for comparison
setkey(before, id_series)
setkey(after, id_series)

# Identify rows where pat_age is different between the two tables, handling NA values
pat_age_diff_na <- before[after,
  on = .(id_series), nomatch = 0,
  .(id_series, pat_bdate, pat_age_before = x.pat_age, pat_age_after = i.pat_age), # Explicitly name pat_age_before as coming from "before"
  by = .EACHI
]

# Filter to show rows where one value is NA and the other is not or the values are simply different
pat_age_diff_na <- pat_age_diff_na[
  (is.na(pat_age_before) & !is.na(pat_age_after)) |
    (!is.na(pat_age_before) & is.na(pat_age_after)) |
    (pat_age_before != pat_age_after)
]

# Print the differences
cat("Rows where pat_age is NA in one table but not in the other, or where the values differ:\n")
print(pat_age_diff_na)


In [ ]:
if (nrow(result) == total_rows) bq_table <- paste0("claims_", year_to_load)

# Check if the table should be dropped and replaced
if (to_drop_bq) {
  tryCatch(
    {
      bq_table_delete(bq_table(gcp_proj, bq_dataset, bq_table))
      message("Table dropped successfully.\n")
    },
    error = function(e) {
      # If the table does not exist, just continue
      if (grepl("Not found", e, ignore.case = TRUE)) {
        message("Table does not exist, nothing to drop.\n")
      } else {
        # If it's a different error, re-throw the error
        stop(e)
      }
    }
  )
}

# Attempt to create the table
tryCatch(
  {
    bq_table_create(
      bq_table(gcp_proj, bq_dataset, bq_table),
      fields = fromJSON(here("data-cleaning/r_scripts", "bq_schema_cleaning.json"), simplifyDataFrame = FALSE)
    )
    skip_bq_upload <<- FALSE
    message("Table created successfully.\n")
  },
  error = function(e) {
    # Check if the error message indicates that the table already exists
    if (grepl("already exists", e, ignore.case = TRUE)) {
      skip_bq_upload <<- TRUE
      message("Table already exists. Skipping creation and upload.")
    } else {
      # If it's a different error, re-throw the error
      stop(e)
    }
  }
)

# Upload to BQ only if table is empty
if (to_bq && !skip_bq_upload) {
  chunk_size <- 1000000 # Adjust the chunk size based on memory availability
  num_chunks <- ceiling(nrow(result) / chunk_size)

  for (i in seq_len(num_chunks)) {
    chunk <- result[((i - 1) * chunk_size + 1):min(i * chunk_size, nrow(result)), ]

    bq_table_upload(
      bq_table(gcp_proj, bq_dataset, bq_table),
      values = chunk,
      write_disposition = if (i == 1) "WRITE_EMPTY" else "WRITE_APPEND"
    )
  }
}


In [ ]:
# in case we want to run this cell independently:
source(here::here("data-cleaning/r_scripts_v2", "03_v2_timing-debug-functions.R"))

# Consolidate all r_scripts scripts into debug.R; useful for debugging
concatenate_r_files(
  here::here("data-cleaning/r_scripts_v2"),
  here::here("data-cleaning/debug/debug-v2.R")
)

if (.Platform$OS.type == "unix") system("cd ~/drg-pipeline && jupyter nbconvert --no-prompt --to script data-cleaning/drg-cleaning-v2.ipynb --output debug/drg-cleaning-v2")


In [3]:
if (TRUE) {
  to_debug <- TRUE
  to_flush_master <- FALSE
  to_flush_partial <- FALSE
}

# Define the paths and corresponding conditions
paths <- list(
  to_flush_master = c(
    "data-cleaning/cache",
    "data-cleaning/data/profvis",
    "data-cleaning/data/aux-files",
    "data-cleaning/data/checkpoints",
    "data-cleaning/debug"
  ),
  to_flush_partial = c(
    "data-cleaning/data/claims/raw/parts",
    "data-cleaning/data/claims/raw/samples"
  )
)

# Iterate over the paths and conditions to delete them if the condition is true
for (condition in names(paths)) {
  if (get(condition)) {
    system(paste(
      "rm -r",
      paste(here::here(unlist(paths[[condition]])), collapse = " ")
    ))
  }
}

if (to_debug) {
  rm(list = ls())
  gc()
}
